# Fig5 — Disease Heritability & Drug Targeting
**Panel A1–A3**: SLDSC enrichment heatmaps (disease / region / stage). (R / ComplexHeatmap)
**Panel B**: Microglia functional cluster AD/PD/IS enrichment barplot. (R / ggplot2)
**Panel C**: Drug–target binary dotplot (Enhanced redox cluster). (Python / matplotlib)


In [ ]:
%%R
# ===========================================================================
# Setup: load libraries, set paths, define shared colour palettes
# ===========================================================================
library(ComplexHeatmap)   # heatmap with annotations & significance stars
library(circlize)         # colour ramp builder (colorRamp2)
library(ggplot2)          # barplot (Panel B)
library(dplyr)            # data wrangling
library(tidyr)            # pivot_wider / pivot_longer
library(purrr)            # pmap for Fisher test
library(tibble)           # column_to_rownames
library(readr)            # read_tsv for SLDSC results
library(grid)             # grid.text for significance stars in heatmap

proj_dir <- Sys.getenv("PROJ_DIR", unset = "path/to/your/project")
setwd(file.path(proj_dir, "data", "human", "output"))
outdir <- "Visualization/Fig5"
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

# ---- Colour palettes shared across all panels ----

# Five major cell lineages for row annotation
cell_category_colors <- c(
  "Glutamatergic neuron" = "#DC9720", "GABAergic neuron" = "#EA7075",
  "Neural precursor cell" = "#7AA8D9", "Glia" = "#BBD642",
  "Non-neural" = "#8E8E90")

# Six microglia functional clusters (Panel B)
function_cluster_colors <- c(
  "Proliferative" = "#D175A9", "Enhanced redox" = "#A09CBF",
  "Inflammatory" = "#C47401", "Lipid associated" = "#4C949F",
  "Surveilling" = "#2A71AA", "Reacting" = "grey")

# Brain region L1 categories (Panel A2 column annotation)
region_l1_colors <- c(
  "Brain" = "#5c8e9b", "Diencephalon" = "#85b76e",
  "Mesencephalon" = "#decf98", "Metencephalon " = "#874e94",
  "Myelencephalon " = "#e97c17", "Prosencephalon" = "#da84b6",
  "Rhombencephalon" = "#4f9446", "Spinal cord" = "#2a71aa",
  "Telencephalon" = "#ae94d7")

# Three broad developmental periods (Panel A3 column annotation)
stage_colors <- c("Prenatal" = "#3953A4", "Postnatal" = "#F89420", "Adult" = "#BE1E2D")

# ---- Helper: map a cell type name to its broad lineage category ----
assign_category <- function(ct) {
  if (ct %in% c("L2/3-CUX2 Glu","L2/3-SATB2 Glu","L4-RORB Glu","L5-BCL11B Glu",
    "L5/6-TLE4 Glu","L5/6-THEMIS Glu","Thalamic_Glu","Hippocampal_Glu",
    "Granule_neuron","Hypothalamic_Glu")) return("Glutamatergic neuron")
  if (ct %in% c("PVALB+ GABA","SST+ GABA","LHX6+LAMP5+ GABA","CALB2+ GABA",
    "VIP+ GABA","RELN+ GABA","Medium_spiny_neuron","Purkinje_neuron",
    "Thalamic_GABA","Midbrain-derived_GABA")) return("GABAergic neuron")
  if (ct %in% c("Astrocyte","Ependymal","Oligodendrocyte")) return("Glia")
  if (ct %in% c("Microglia","Macrophage","Fibroblast","Endothelial_cell",
    "Pericyte","Choroid_plexus_epithelial","Vascular_smooth_muscle_cell")) return("Non-neural")
  if (ct %in% c("Glioblast","Neuroblast","Neuronal_IPC","Radial_glia",
    "Oligodendrocyte_precursor_committed","Oligodendrocyte_precursor")) return("Neural precursor cell")
  return("Non-neural")
}

In [ ]:
%%R
# ===========================================================================
# Load SLDSC results & reference metadata
# ===========================================================================
# SLDSC merged output: cell type × disease -log10(p) enrichment matrix
sldsc_file <- file.path("Analysis","disease","output","celltype_sldsc_analysis",
  "merged_sldsc_mic_func6_results_new.tsv")
sldsc <- read_tsv(sldsc_file, show_col_types = FALSE)

# Disease/trait list for the heatmap columns
target_diseases <- c("ADHD","AD","ALS","ANX","ASD","ASP","AUD","BIP","CI",
  "DPW","EA","EPI","INS","MDD","MS","PD","SCZ","TS","NEU","IQ")

# Reference atlas metadata (used by Panels A2–A3 for OR enrichment)
meta <- read.csv("Annotation_hvg5000/final/hs_all_metadata_final.csv", header=TRUE)
rownames(meta) <- meta$cell_name; meta <- meta[meta$type_2 != "Remove",]
total_cells <- nrow(meta)

### Panel A1: Disease Enrichment Heatmap


In [ ]:
%%R
# ===========================================================================
# Panel A1: Cell type × Disease enrichment heatmap
# Data source: SLDSC -log10(p) values per (cell type, disease)
# Stars = p < 0.05; row annotation = cell lineage category
# ===========================================================================
# Filter to target diseases and compute -log10(p)
celltype_disease <- sldsc %>% 
  filter(Disease %in% target_diseases) %>%
  mutate(neg_log10p=as.numeric(log10p), cell_category=sapply(Cell_Type,assign_category)) %>%
  filter(!is.na(neg_log10p))

# Build wide matrix: rows=cell types, columns=diseases
heatmap_disease <- celltype_disease %>%
  select(Cell_Type,Disease,neg_log10p) %>%
  pivot_wider(names_from=Disease,values_from=neg_log10p,values_fill=0) %>%
  column_to_rownames("Cell_Type") %>% as.matrix()

# Significance star matrix (same dimensions)
signif_disease <- matrix("",nrow=nrow(heatmap_disease),ncol=ncol(heatmap_disease))
signif_disease[heatmap_disease > -log10(0.05)] <- "*"
dimnames(signif_disease) <- dimnames(heatmap_disease)

# Row annotation: colour bar by cell lineage category
row_cats <- sapply(rownames(heatmap_disease),assign_category)
row_ha <- rowAnnotation(Category=row_cats,col=list(Category=cell_category_colors),
  show_legend=TRUE,simple_anno_size=unit(0.2,"cm"))

# Draw heatmap: white → pink → dark red
ht_a1 <- Heatmap(heatmap_disease,name="-log10(P)",
  col=colorRamp2(c(0,max(heatmap_disease)/2,max(heatmap_disease)),
    c("white","#F6BDC0","#781426")),
  left_annotation=row_ha,row_names_gp=gpar(fontsize=10),
  column_names_gp=gpar(fontsize=12,fontface="bold"),
  column_title="Disease/Trait",row_title="Cell Class",
  cluster_rows=FALSE,cluster_columns=FALSE,
  cell_fun=function(j,i,x,y,w,h,fill){if(signif_disease[i,j]!="")grid.text(signif_disease[i,j],x,y,gp=gpar(col="black",fontsize=12,fontface="bold"))},
  heatmap_legend_param=list(title="-log10(P)",direction="horizontal"))
pdf(file.path(outdir,"celltype_disease_enrichment_heatmap.pdf"),width=20,height=10)
draw(ht_a1,heatmap_legend_side="right",annotation_legend_side="right",merge_legend=TRUE,padding=unit(c(20,2,2,2),"mm"))
dev.off(); message("Panel A1 saved")

### Panel A2: Region Enrichment Heatmap


In [ ]:
%%R
# ===========================================================================
# Panel A2: Cell type × Brain Region enrichment heatmap
# Method: Odds Ratio from Fisher exact test per (cell type, region)
# Colour: log2(OR), blue (depleted) → white → red (enriched)
# Columns sorted by Region_L1, annotated at bottom
# ===========================================================================
# Count totals per cell type and per region
type_totals <- meta %>% count(type_2) %>% rename(total_type=n)
region_totals <- meta %>% count(region_L2) %>% rename(total_region=n)

# Compute log2(OR) for each (cell type, region) pair
# OR = (a/b) / (c/d) where a=count, b=type_total-a, c=region_total-a, d=rest
region_or <- meta %>% count(type_2,region_L2) %>%
  left_join(region_totals,by="region_L2") %>% left_join(type_totals,by="type_2") %>%
  mutate(or_val=(n/(total_type-n+1))/((total_region-n+1)/(total_cells-n+1)),
         log_or=log2(pmin(pmax(or_val,0.5),2)))  # clip to [0.5, 2] for stability

# Pivot to matrix and sort columns by Region_L1
region_mat <- region_or %>% select(type_2,region_L2,log_or) %>%
  pivot_wider(names_from=region_L2,values_from=log_or,values_fill=0) %>%
  column_to_rownames("type_2") %>% as.matrix()
region_map <- meta %>% select(region_L2,region_L1) %>% distinct()
l1_order <- region_map$region_L1[match(colnames(region_mat),region_map$region_L2)]
region_mat <- region_mat[,order(l1_order),drop=FALSE]

# Column annotation: region L1 colour bar
col_ha <- HeatmapAnnotation(Region_L1=l1_order[order(l1_order)],
  col=list(Region_L1=region_l1_colors),show_legend=TRUE,show_annotation_name=FALSE)

ht_a2 <- Heatmap(region_mat,name="log2(OR)",
  col=colorRamp2(c(-1,0,1),c("#2166ac","white","#b2182b")),
  left_annotation=row_ha,bottom_annotation=col_ha,row_names_gp=gpar(fontsize=10),
  column_names_gp=gpar(fontsize=12,fontface="bold"),
  column_title="Cell Class",row_title="Brain Region",
  cluster_rows=FALSE,cluster_columns=FALSE,
  heatmap_legend_param=list(title="log2(OR)",direction="horizontal"))
pdf(file.path(outdir,"celltype_region_enrichment_heatmap.pdf"),width=20,height=10)
draw(ht_a2,heatmap_legend_side="right",annotation_legend_side="right",
  merge_legend=TRUE,padding=unit(c(20,2,2,2),"mm"))
dev.off(); message("Panel A2 saved")

### Panel A3: Stage Enrichment Heatmap


In [ ]:
%%R
# ===========================================================================
# Panel A3: Cell type × Developmental Stage enrichment heatmap
# Method: same log2(OR) approach as A2, but rows = stages
# Columns annotated by broad period (Prenatal / Postnatal / Adult)
# ===========================================================================
stage_totals <- meta %>% count(stage) %>% rename(total_stage=n)
stage_or <- meta %>% count(type_2,stage) %>%
  left_join(stage_totals,by="stage") %>% left_join(type_totals,by="type_2") %>%
  mutate(or_val=(n/(total_type-n+1))/((total_stage-n+1)/(total_cells-n+1)),
         log_or=log2(pmin(pmax(or_val,0.5),2)))

stage_mat <- stage_or %>% select(type_2,stage,log_or) %>%
  pivot_wider(names_from=stage,values_from=log_or,values_fill=0) %>%
  column_to_rownames("type_2") %>% as.matrix()

# Classify stages into three broad periods
stage_l1 <- ifelse(grepl("^0[1-7]",colnames(stage_mat)),"Prenatal",
  ifelse(grepl("^0[8-9]|^1[0-2]",colnames(stage_mat)),"Postnatal","Adult"))
stage_anno <- HeatmapAnnotation(Phase=stage_l1,col=list(Phase=stage_colors),
  show_legend=TRUE,show_annotation_name=FALSE)

ht_a3 <- Heatmap(stage_mat,name="log2(OR)",
  col=colorRamp2(c(-1,0,1),c("#2166ac","white","#b2182b")),
  left_annotation=row_ha,bottom_annotation=stage_anno,row_names_gp=gpar(fontsize=10),
  column_names_gp=gpar(fontsize=12,fontface="bold"),
  column_title="Cell Class",row_title="Developmental Stage",
  cluster_rows=FALSE,cluster_columns=FALSE,
  heatmap_legend_param=list(title="log2(OR)",direction="horizontal"))
pdf(file.path(outdir,"celltype_stage_enrichment_heatmap.pdf"),width=20,height=10)
draw(ht_a3,heatmap_legend_side="right",annotation_legend_side="right",
  merge_legend=TRUE,padding=unit(c(20,2,2,2),"mm"))
dev.off(); message("Panel A3 saved")

### Panel B: Microglia Functional Cluster Barplot


In [ ]:
%%R
# ===========================================================================
# Panel B: Microglia functional cluster enrichment barplot (AD / PD / IS)
# Data: SLDSC enrichment values for 6 functional clusters across 3 diseases
# Stars: *** p<0.001, ** p<0.01, * p<0.05; dashed line at enrichment = 1
# ===========================================================================
func6_data <- sldsc %>% 
  filter(Cell_Type %in% names(function_cluster_colors),
         Disease %in% c("AD","PD","IS")) %>%
  mutate(
    Enrichment   = as.numeric(Enrichment_num),
    Enrichment_p = as.numeric(Enrichment_p),
    # Assign significance stars
    significance = case_when(
      Enrichment_p < 0.001 ~ "***",
      Enrichment_p < 0.01  ~ "**",
      Enrichment_p < 0.05  ~ "*",
      TRUE ~ ""),
    Cell_Type = factor(Cell_Type, levels = names(function_cluster_colors))) %>%
  filter(!is.na(Enrichment))

p_b <- ggplot(func6_data, aes(x = Disease, y = Enrichment, fill = Cell_Type)) +
  geom_col(position = position_dodge(width = 0.8), width = 0.7) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "gray50", linewidth = 0.8) +
  # Star labels at top of each bar
  geom_text(aes(label = significance, y = Enrichment + max(Enrichment) * 0.01),
            position = position_dodge(width = 0.8), vjust = 0, size = 4, fontface = "bold") +
  scale_fill_manual(values = function_cluster_colors) +
  labs(x = "Disease", y = "Enrichment", fill = "Microglia Clusters") +
  theme_minimal() +
  theme(axis.title = element_text(size = 12, face = "bold"),
        axis.text  = element_text(size = 10),
        legend.position = "right",
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.8),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank())

ggsave(file.path(outdir, "microglia_func6_enrichment_barplot.pdf"),
       p_b, width = 10, height = 5)
message("Panel B saved")

### Panel C: Drug–Target Dotplot (Python / matplotlib)


In [ ]:
# ===========================================================================
# Panel C: Drug–Target binary dotplot (Enhanced redox microglia cluster)
# Reads a binary matrix (drug × target gene: 1=hit, 0=no hit)
# Each dot = one drug–gene association; Galantamine & IDH1 highlighted
# ===========================================================================
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

proj_dir = os.environ.get("PROJ_DIR", "path/to/your/project")
data_dir = os.path.join(proj_dir, "data", "human", "output")
out_dir  = os.path.join(data_dir, "Visualization", "Fig5")
os.makedirs(out_dir, exist_ok=True)

# Binary matrix: rows = drugs, columns = target molecules
drug_file = os.path.join(data_dir, "Analysis", "disease", "drug", "mic",
                         "enhanced_redox_drug_target_matrix.csv")

if os.path.exists(drug_file):
    mat = pd.read_csv(drug_file, index_col=0)
    
    # Auto-sized figure: width ~ number of targets, height ~ number of drugs
    fig, ax = plt.subplots(figsize=(0.8 * mat.shape[1] + 0.5,
                                    0.25 * mat.shape[0] + 0.5))
    
    # Draw a dot for every (=1) cell in the matrix
    for i, drug in enumerate(mat.index):
        for j, gene in enumerate(mat.columns):
            if mat.loc[drug, gene] == 1:
                ax.scatter(j, i, s=180, color="darkred", alpha=0.8)
    
    # Axis labels
    ax.set_xticks(range(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=90)
    ax.set_yticks(range(mat.shape[0]))
    ax.set_yticklabels(mat.index)
    ax.set_ylim(-0.6, mat.shape[0] - 0.4)
    
    # Highlight specific drug and gene of interest
    for t in ax.get_yticklabels():
        if t.get_text() == "Galantamine hydrobromide":
            t.set_color("darkred"); t.set_fontweight("bold")
    for t in ax.get_xticklabels():
        if t.get_text() == "IDH1":
            t.set_color("darkred"); t.set_fontweight("bold")
    
    ax.set_xlabel("Target Molecules")
    ax.set_ylabel("Drugs")
    ax.invert_yaxis()          # first drug at top
    plt.tight_layout()
    fig.savefig(os.path.join(out_dir, "enhanced_redox_drug_target_dotplot.pdf"),
                dpi=300)
    plt.close()
    print("Panel C saved")
else:
    print(f"Drug matrix not found: {drug_file}")